In [1]:
#Import essential libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import concat_ws, col, isnan, isnull, when, count, round, date_format, countDistinct, max
from pyspark.sql.types import DecimalType

#Initiate Spark Session
spark = SparkSession.builder \
    .appName("Optasia Assignment") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.7.3") \
    .getOrCreate()

spark

In [2]:
#Read and load 2nd CSV file. Use seperator and inferSchema to recognize data types. 
transactions_df = spark.read.csv("../01.Data/data_transactions.csv", sep = ',', inferSchema=True)

In [3]:
transactions_df.show(5)

+--------------------+--------+----+---+
|                 _c0|     _c1| _c2|_c3|
+--------------------+--------+----+---+
|2021-08-16 00:14:...|123456.0| 0.3|SMS|
|2021-08-16 00:54:...|123451.0|0.15|SMS|
|2021-08-16 00:04:...|123452.0|0.15|SMS|
|2021-08-16 00:39:...|123453.0|0.15|SMS|
|2021-08-16 00:14:...|123454.0| 0.3|SMS|
+--------------------+--------+----+---+
only showing top 5 rows



In [4]:
#Check data types. Data types recognized are not the requested.
transactions_df.dtypes

[('_c0', 'string'), ('_c1', 'double'), ('_c2', 'string'), ('_c3', 'string')]

In [5]:
#Check initial database lines
transactions_df.count()

106

In [6]:
#Initial database has no headers. Column context and names are given in the instructions
transactions_df = transactions_df.toDF("timestamp", "subscriber_id", "amount", "channel")
transactions_df.show(5)

+--------------------+-------------+------+-------+
|           timestamp|subscriber_id|amount|channel|
+--------------------+-------------+------+-------+
|2021-08-16 00:14:...|     123456.0|   0.3|    SMS|
|2021-08-16 00:54:...|     123451.0|  0.15|    SMS|
|2021-08-16 00:04:...|     123452.0|  0.15|    SMS|
|2021-08-16 00:39:...|     123453.0|  0.15|    SMS|
|2021-08-16 00:14:...|     123454.0|   0.3|    SMS|
+--------------------+-------------+------+-------+
only showing top 5 rows



In [7]:
#Correct the data types format before we begin sanity & correctness tests
transactions_df = transactions_df.withColumn("timestamp", col("timestamp").cast("timestamp")) \
                                 .withColumn("subscriber_id", col("subscriber_id").cast("int")) \
                                 .withColumn("amount", col("amount").cast(DecimalType(10, 4)))
transactions_df.dtypes

[('timestamp', 'timestamp'),
 ('subscriber_id', 'int'),
 ('amount', 'decimal(10,4)'),
 ('channel', 'string')]

In [8]:
transactions_df.show(5)

+-------------------+-------------+------+-------+
|          timestamp|subscriber_id|amount|channel|
+-------------------+-------------+------+-------+
|2021-08-15 23:14:01|       123456|0.3000|    SMS|
|2021-08-15 23:54:43|       123451|0.1500|    SMS|
|2021-08-15 23:04:29|       123452|0.1500|    SMS|
|2021-08-15 23:39:05|       123453|0.1500|    SMS|
|2021-08-15 23:14:19|       123454|0.3000|    SMS|
+-------------------+-------------+------+-------+
only showing top 5 rows



In [9]:
#Check for null values, in this case there are in 3 columns
transactions_df.select([
    count(when(col(c).isNull(), c)).alias(c + "_nulls")
    for c in transactions_df.columns
]).show()

+---------------+-------------------+------------+-------------+
|timestamp_nulls|subscriber_id_nulls|amount_nulls|channel_nulls|
+---------------+-------------------+------------+-------------+
|              2|                  0|           2|            2|
+---------------+-------------------+------------+-------------+



In [10]:
#Make the decision to drop lines with null values
transactions_df=transactions_df.dropna()

In [11]:
#Rechecking database lines after dropna
transactions_df.count()

102

In [12]:
#Check if ids are unique. In this case they are. Duplicate ids would not necessarily indicate wrong entries as this is a transactions database.
transactions_df.select(countDistinct("subscriber_id")).show()

+-----------------------------+
|count(DISTINCT subscriber_id)|
+-----------------------------+
|                          102|
+-----------------------------+



In [13]:
#Read and load "unmatched_transactions". This table could preexist. In this case it is created for the first time after the databases are joined.
#After merging the databases duplicate entries are dropped.
#In this case all entries preexist in our transactions database so no entry will be added. (they will be added and dropped)
#A message is printed to indicate if the unmatched entries file is found in the postgres database and the union is iniated
#When running the code for the first time no file will be found as it is created later in the notebook
try:
    unmatched_previous_df = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://optasia_postgres:5432/optasia_db") \
    .option("dbtable", "unmatched_transactions") \
    .option("user", "optasia_user") \
    .option("password", "optasia_pass") \
    .option("driver", "org.postgresql.Driver") \
    .load()
    
    transactions_df = transactions_df.unionByName(unmatched_previous_df).dropDuplicates(["subscriber_id", "timestamp", "amount", "channel"])
    
    print("Previous unmatched entries successfully unioned")
except:
    print("No previous unmatched entries found in Postgres")


Previous unmatched entries successfully unioned


In [14]:
#Recheck database lines
transactions_df.count()

102

In [15]:
#Read and load the subscribers table from the postgres database
subscribers_final_df = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://optasia_postgres:5432/optasia_db") \
    .option("dbtable", "subscribers") \
    .option("user", "optasia_user") \
    .option("password", "optasia_pass") \
    .option("driver", "org.postgresql.Driver") \
    .load()

In [16]:
#Inner join the two tables on the subscriber id which is common.
#Inner join because we want to keep only transactions that have a match in the subscribers database
joined_df = transactions_df.join(
    subscribers_final_df,
    transactions_df.subscriber_id == subscribers_final_df.sub_id,
    how="inner")

In [17]:
joined_df.show(5)

+-------------------+-------------+------+-------+---------------+------+----------+
|          timestamp|subscriber_id|amount|channel|        row_key|sub_id|    act_dt|
+-------------------+-------------+------+-------+---------------+------+----------+
|2021-08-16 07:16:02|       987654|0.7500|    SMS|987654_20210709|987654|2021-07-09|
|2021-08-16 05:08:18|       543215|0.7500|    SMS|543215_20201222|543215|2020-12-22|
|2021-08-16 06:32:02|       543219|0.1500|    SMS|543219_20201222|543219|2020-12-22|
|2021-08-15 23:06:27|       234595|1.5000|    SMS|234595_20210810|234595|2021-08-10|
|2021-08-16 05:29:18|       345992|0.1500|    SMS|345992_20210709|345992|2021-07-09|
+-------------------+-------------+------+-------+---------------+------+----------+
only showing top 5 rows



In [18]:
#Left join the two tables and filter transactions that have no match on the subscriber id to create the unmatched database.

unmatched_df = transactions_df.join(
    subscribers_final_df,
    transactions_df.subscriber_id == subscribers_final_df.sub_id,
    how="left"
).filter(subscribers_final_df["sub_id"].isNull())

#Save unmatched entries in the postgres database for rechecking later with another batch of transactions
#We keep only the original columns so they can be "merged" with the transactions table.

unmatched_df.select("timestamp", "subscriber_id", "amount", "channel") \
    .write \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://optasia_postgres:5432/optasia_db") \
    .option("dbtable", "unmatched_transactions") \
    .option("user", "optasia_user") \
    .option("password", "optasia_pass") \
    .option("driver", "org.postgresql.Driver") \
    .mode("overwrite") \
    .save()

In [19]:
unmatched_df.show(5)

+-------------------+-------------+------+-------+-------+------+------+
|          timestamp|subscriber_id|amount|channel|row_key|sub_id|act_dt|
+-------------------+-------------+------+-------+-------+------+------+
|2021-08-16 07:10:34|       459995|0.1500|    SMS|   NULL|  NULL|  NULL|
|2021-08-16 05:49:37|       459999|0.3000|    SMS|   NULL|  NULL|  NULL|
|2021-08-15 23:04:44|       654324|0.3000|    SMS|   NULL|  NULL|  NULL|
|2021-08-16 00:42:31|       543210|0.7500|    SMS|   NULL|  NULL|  NULL|
|2021-08-15 23:15:23|       459993|0.3000|    SMS|   NULL|  NULL|  NULL|
+-------------------+-------------+------+-------+-------+------+------+
only showing top 5 rows



In [20]:
#Create the transactions final table as requested for this assignment (columns, names, formating)
result_df = joined_df.select(
    col("timestamp"),
    col("row_key").alias("sub_row_key"),
    col("sub_id"),
    col("amount"),
    col("channel"),
    date_format("act_dt", "yyyyMMdd").alias("activation_date")
)

In [21]:
result_df.show(5)

+-------------------+---------------+------+------+-------+---------------+
|          timestamp|    sub_row_key|sub_id|amount|channel|activation_date|
+-------------------+---------------+------+------+-------+---------------+
|2021-08-16 07:16:02|987654_20210709|987654|0.7500|    SMS|       20210709|
|2021-08-16 05:08:18|543215_20201222|543215|0.7500|    SMS|       20201222|
|2021-08-16 06:32:02|543219_20201222|543219|0.1500|    SMS|       20201222|
|2021-08-15 23:06:27|234595_20210810|234595|1.5000|    SMS|       20210810|
|2021-08-16 05:29:18|345992_20210709|345992|0.1500|    SMS|       20210709|
+-------------------+---------------+------+------+-------+---------------+
only showing top 5 rows



In [22]:
#Save in a parquet file in the designated folder (declared as a pyspark volume in the docker file) 
#If the file already exists overwrite it
result_df.write \
    .mode("overwrite") \
    .parquet("/home/jovyan/parquet_files/transactions_enriched.parquet")

In [23]:
#Check that the file is saved successfully
check_df = spark.read.parquet("/home/jovyan/parquet_files/transactions_enriched.parquet")
check_df.show()

+-------------------+---------------+------+------+-------+---------------+
|          timestamp|    sub_row_key|sub_id|amount|channel|activation_date|
+-------------------+---------------+------+------+-------+---------------+
|2021-08-16 07:16:02|987654_20210709|987654|0.7500|    SMS|       20210709|
|2021-08-16 05:08:18|543215_20201222|543215|0.7500|    SMS|       20201222|
|2021-08-16 06:32:02|543219_20201222|543219|0.1500|    SMS|       20201222|
|2021-08-15 23:06:27|234595_20210810|234595|1.5000|    SMS|       20210810|
|2021-08-16 05:29:18|345992_20210709|345992|0.1500|    SMS|       20210709|
|2021-08-15 23:14:25|739280_20201222|739280|0.3000|    SMS|       20201222|
|2021-08-16 08:54:26|876543_20201222|876543|0.1500|    SMS|       20201222|
|2021-08-15 23:06:21|123459_20210810|123459|0.1500|    SMS|       20210810|
|2021-08-16 06:19:31|543218_20201222|543218|0.7500|    SMS|       20201222|
|2021-08-16 03:33:53|987658_20210709|987658|0.3000|    SMS|       20210709|
|2021-08-16 

In [24]:
check_df.count()

75